In [ ]:
!git clone https://github.com/bowang-lab/MedSAM2.git
%cd /content/MedSAM2
!pip install -e ".[dev]"
!bash download.sh
%cd /content

Cloning into 'MedSAM2'...
remote: Enumerating objects: 297, done.
remote: Counting objects: 100% (165/165), done.
remote: Compressing objects: 100% (101/101), done.
remote: Total 297 (delta 91), reused 64 (delta 64), pack-reused 132 (from 1)
Receiving objects: 100% (297/297), 18.82 MiB | 27.60 MiB/s, done.
Resolving deltas: 100% (122/122), done.
/content/MedSAM2
Obtaining file:///content/MedSAM2
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 2.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.6/74.6 kB 7.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 5.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 9.0 MB/s eta

In [ ]:
import os, gzip, shutil, torch
from pathlib import Path

# 1. Download Dataset
!hf download wanglab/LLD-MMRI-MedSAM2 --repo-type dataset --local-dir /content/MedSAM2/LLD-MMRI-MedSAM2

# 2. Extraction Function
def extract_and_clean(folder_path):
    directory = Path(folder_path)
    # Target .nii.gz files
    for gz_file in directory.glob("*.nii.gz"):
        nii_file = gz_file.with_suffix('')
        print(f"Extracting {gz_file.name}...")
        with gzip.open(gz_file, 'rb') as f_in, open(nii_file, 'wb') as f_out:
            shutil.copyfileobj(f_in, f_out)
        gz_file.unlink() # Delete .gz after extraction

# Apply to images and labels
extract_and_clean("/content/MedSAM2/LLD-MMRI-MedSAM2/images")
extract_and_clean("/content/MedSAM2/LLD-MMRI-MedSAM2/labels")

Streaming output truncated to the last 5000 lines.
Extracting MR27202_1_C+A_0000.nii.gz...
Extracting MR180939_0_C-pre_0000.nii.gz...
Extracting MR70480_1_C-pre_0000.nii.gz...
Extracting MR200356_2_C+V_0000.nii.gz...
Extracting MR198874_6_OutPhase_0000.nii.gz...
Extracting MR146023_5_DWI_0000.nii.gz...
Extracting MR123588_6_C+V_0000.nii.gz...
Extracting MR132628_6_C+V_0000.nii.gz...
Extracting MR199345_1_C-pre_0000.nii.gz...
Extracting MR22508_1_T2WI_0000.nii.gz...
Extracting MR233963_2_T2WI_0000.nii.gz...
Extracting MR209135_1_C-pre_0000.nii.gz...
Extracting MR146979_1_DWI_0000.nii.gz...
Extracting MR175609_0_OutPhase_0000.nii.gz...
Extracting MR120289_6_T2WI_0000.nii.gz...
Extracting MR237152_3_C-pre_0000.nii.gz...
Extracting MR203324_0_C-pre_0000.nii.gz...
Extracting MR-467988_5_C+V_0000.nii.gz...
Extracting MR226253_5_C+A_0000.nii.gz...
Extracting MR53725_6_C-pre_0000.nii.gz...
Extracting MR8888_6_C-pre_0000.nii.gz...
Extracting MR222125_3_T2WI_0000.nii.gz...
Extracting MR176611_0_

OSError: [Errno 28] No space left on device

In [ ]:
import json

# 1. Load the JSON file
json_path = '/content/MedSAM2/LLD-MMRI-MedSAM2/LLD_MMRI_Annotation.json'
with open(json_path, 'r') as f:
    json_data = json.load(f)

# 2. Extract the patient IDs (keys under 'Annotation_info')
patient_ids = list(json_data['Annotation_info'].keys())

# 3. Print the results
print("=========================================")
print(f"📊 Total unique Patient IDs in JSON: {len(patient_ids)}")
print("=========================================")

# Show the first 10 patient IDs as a quick preview
print("First 10 Patient IDs:", patient_ids[:10])

In [ ]:
import os
from collections import Counter

# Get all filenames
files = os.listdir('/content/MedSAM2/LLD-MMRI-MedSAM2/images')

# Extract the 'PatientID_LesionID' part (everything before the phase)
# Example: MR-391135_1_C+A -> MR-391135_1
identifiers = [f.split('_')[0] + "_" + f.split('_')[1] for f in files if f.endswith('.nii')]

# Count how many times each 'Patient_Lesion' identifier appears
counts = Counter(identifiers)

# Are there any identifiers that appear more than, say, 8 times?
# (Since you have 8 phases, a single lesion should appear exactly 8 times total)
duplicates = {k: v for k, v in counts.items() if v > 8}

print(f"Total unique Patient_Lesion combinations: {len(counts)}")
if len(duplicates) > 0:
    print(f"Warning: Found Patient_Lesions with more than 8 files: {duplicates}")
else:
    print("Everything looks clean! Each tumor has exactly 8 phases.")

In [ ]:
import json, re, sys, torch, numpy as np, pandas as pd, SimpleITK as sitk, matplotlib.pyplot as plt
from glob import glob
from os.path import join, basename
from skimage import measure
from PIL import Image
from collections import OrderedDict

sys.path.append('/content/MedSAM2')
from sam2.build_sam import build_sam2_video_predictor_npz

# Fix seeds
torch.set_float32_matmul_precision('high')
torch.manual_seed(2024)
np.random.seed(2024)

def getLargestCC(segmentation):
    labels = measure.label(segmentation)
    return labels == np.argmax(np.bincount(labels.flat)[1:]) + 1 if labels.max() > 0 else segmentation

def calculate_dice(pred, target):
    p, t = np.asarray(pred).astype(bool), np.asarray(target).astype(bool)
    if p.sum() == 0 and t.sum() == 0: return 1.0
    return 2.0 * np.logical_and(p, t).sum() / (p.sum() + t.sum())

def resize_for_medsam(array, size=512):
    d, h, w = array.shape
    out = np.zeros((d, 3, size, size))
    for i in range(d):
        img = Image.fromarray(array[i].astype(np.uint8)).convert("RGB").resize((size, size))
        out[i] = np.array(img).transpose(2, 0, 1)
    return out

In [ ]:
checkpoint = '/content/MedSAM2/checkpoints/MedSAM2_latest.pt'
imgs_path = '/content/MedSAM2/LLD-MMRI-MedSAM2/images'
labels_path = '/content/MedSAM2/LLD-MMRI-MedSAM2/labels'
pred_save_dir = "/content/LLD_MMRI_results"
os.makedirs(pred_save_dir, exist_ok=True)

with open('/content/MedSAM2/LLD-MMRI-MedSAM2/LLD_MMRI_Annotation.json', 'r') as f:
    annotation_info = json.load(f)['Annotation_info']

predictor = build_sam2_video_predictor_npz("configs/sam2.1_hiera_t512.yaml", checkpoint)
seg_info = OrderedDict({'nii_name': [], 'studyUID': [], 'phase': [], 'key_slice_index': [], 'dice_score': []})
phase_map = {'InPhase': 'In Phase', 'OutPhase': 'Out Phase', 'C+A': 'C+A', 'C+V': 'C+V', 'C+Delay': 'C+Delay', 'C-pre': 'C-pre', 'DWI': 'DWI', 'T2WI': 'T2WI'}

for nii_path in sorted(glob(join(imgs_path, '*.nii'))):
    nii_fname = basename(nii_path)
    match = re.match(r'(MR-?\d+|MR\d+)_(\d+)_(.+)_0000\.nii', nii_fname)
    if not match: continue
    pid, lid, phase_str = match.groups()
    gt_path = join(labels_path, nii_fname)

    if not os.path.exists(gt_path) or pid not in annotation_info: continue
    record = next((r for r in annotation_info[pid] if r['phase'] == phase_map.get(phase_str, phase_str)), None)
    if not record: continue

    study_uid = record['studyUID'] # Extracted studyUID
    best_box = max(record['annotation']['lesion']['0']['bbox']['2D_box'], key=lambda x: x['area'])
    bbox = np.array([best_box['x_min'], best_box['y_min'], best_box['x_max'], best_box['y_max']])
    key_idx = int(best_box['slice_idx'])

    try:
        # Load and Preprocess
        nii_data = sitk.GetArrayFromImage(sitk.ReadImage(nii_path))
        gt_data = (sitk.GetArrayFromImage(sitk.ReadImage(gt_path)) > 0).astype(np.uint8)
        norm = ((np.clip(nii_data, np.percentile(nii_data, 1), np.percentile(nii_data, 99)) - np.percentile(nii_data, 1)) / (np.percentile(nii_data, 99) - np.percentile(nii_data, 1)) * 255).astype(np.uint8)

        # Inference
        img_tensor = torch.from_numpy(resize_for_medsam(norm) / 255.0).float().cuda()
        with torch.inference_mode(), torch.autocast("cuda", dtype=torch.bfloat16):
            state = predictor.init_state(img_tensor, norm.shape[1], norm.shape[2])
            _, _, logits = predictor.add_new_points_or_box(state, frame_idx=key_idx, obj_id=1, box=bbox)
            segs = np.zeros(nii_data.shape, dtype=np.uint8)
            for f_idx, _, l in predictor.propagate_in_video(state):
                segs[f_idx] = (l[0] > 0.0).cpu().numpy()
            for f_idx, _, l in predictor.propagate_in_video(state, reverse=True):
                segs[f_idx] = (l[0] > 0.0).cpu().numpy()
            predictor.reset_state(state)

        # Evaluate
        segs = getLargestCC(segs)
        dice = calculate_dice(segs, gt_data)

        # VISUALIZATION (Clean overlay using show_mask helper)
        fig, axes = plt.subplots(1, 3, figsize=(18, 6))

        # 1. Raw Input Image
        axes[0].imshow(norm[key_idx], cmap='gray')
        axes[0].set_title('Input Image', fontsize=14)
        axes[0].axis('off')

        # 2. Input + Ground Truth (Green)
        axes[1].imshow(norm[key_idx], cmap='gray')
        show_mask(gt_data[key_idx], ax=axes[1], mask_color=np.array([0, 1, 0])) # Sharp green overlay
        axes[1].set_title('Ground Truth (Green)', fontsize=14)
        axes[1].axis('off')

        # 3. Input + Prediction (Yellow)
        axes[2].imshow(norm[key_idx], cmap='gray')
        show_mask(segs[key_idx], ax=axes[2], mask_color=np.array([251/255, 252/255, 30/255])) # Sharp yellow overlay
        axes[2].set_title(f'Generated Mask (Yellow)\nDice Score: {dice:.4f}', fontsize=14)
        axes[2].axis('off')

        plt.tight_layout()
        plt.savefig(join(pred_save_dir, nii_fname.replace('.nii', '_comparison.png')), bbox_inches='tight')
        plt.close()

        # TRACK RESULTS
        save_seg_name = nii_fname.replace('.nii', f'_k{key_idx}_mask.nii')
        sitk_mask = sitk.GetImageFromArray(segs)
        sitk_mask.CopyInformation(sitk.ReadImage(nii_path))
        sitk.WriteImage(sitk_mask, join(pred_save_dir, save_seg_name))

        seg_info['nii_name'].append(save_seg_name)
        seg_info['studyUID'].append(study_uid)
        seg_info['phase'].append(phase_map.get(phase_str, phase_str))
        seg_info['key_slice_index'].append(key_idx)
        seg_info['dice_score'].append(dice)
        print(f"Processed {nii_fname}, Dice: {dice:.4f}")

    except Exception as e:
        print(f" -> Error processing {nii_fname}: {e}")
        plt.close('all')

# Save tracking results to DataFrame
seg_info_df = pd.DataFrame(seg_info)
results_csv_path = join(pred_save_dir, "lld_mmri_final_dice_results.csv")
seg_info_df.to_csv(results_csv_path, index=False)

# NEW: Print final metrics and output directory path
print("\n==================================================")
print("🎉 Evaluation Complete!")
print(f"📁 Total processed valid cases: {len(seg_info_df)}")
if len(seg_info_df) > 0:
    print(f"🏆 Mean Dice Score across dataset: {seg_info_df['dice_score'].mean():.4f}")
print(f"📍 Comparison PNGs are saved at: {pred_save_dir}")
print(f"📄 Results spreadsheet saved at: {results_csv_path}")
print("==================================================")

# Show preview
display(seg_info_df.head())